In [ ]:
import json
import re
from datetime import datetime

# -------- Helpers --------

def extract_date(posted_on):
    """
    Extracts date from 'Posted on February 06, 2026 | ...'
    and converts to MM/DD/YYYY
    """
    match = re.search(r"Posted on ([A-Za-z]+ \d{2}, \d{4})", posted_on)
    if not match:
        return None

    date_str = match.group(1)
    return datetime.strptime(date_str, "%B %d, %Y").strftime("%m/%d/%Y")


In [ ]:

def extract_city(title):
    """
    Look for 'City of' and return the very next word
    """
    match = re.search(r"City of\s+(\w+)", title)
    return match.group(1) if match else None


def extract_county(title):
    """
    Look for 'County' and return the word immediately before it
    """
    match = re.search(r"(\w+)\s+County", title)
    return f"{match.group(1)} County" if match else None





In [ ]:
import os
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# -------- Main Processing --------

INPUT_FOLDER = "extracted_json"
OUTPUT_FOLDER = "processed_json"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def process_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data:
        title = item.get("title", "")
        posted_on = item.get("posted_on", "")

        issued_date = None
        rescinded_date = None

        if "Issued for" in title:
            issued_date = extract_date(posted_on)

        if "Rescinded for" in title:
            rescinded_date = extract_date(posted_on)

        item["issued_date"] = issued_date
        item["rescinded_date"] = rescinded_date
        item["city"] = extract_city(title)
        item["county"] = extract_county(title)

    output_path = Path(OUTPUT_FOLDER) / Path(file_path).name

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

# Process all JSON files in folder
for file_name in os.listdir(INPUT_FOLDER):
    if file_name.endswith(".json"):
        process_file(os.path.join(INPUT_FOLDER, file_name))



files = [
    os.path.join(INPUT_FOLDER, f)
    for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".json")
]

with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(process_file, files)